<a href="https://colab.research.google.com/github/vaish2814/FastAPI_Food_Delivery_Project/blob/main/FastAPI_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn nest-asyncio pyngrok pydantic

In [2]:
import nest_asyncio
from fastapi import FastAPI, HTTPException, Query, Path, status
from pydantic import BaseModel, Field
from typing import Optional, List
import uvicorn

# 1. Setup for Colab
nest_asyncio.apply()
app = FastAPI(title="Final Project: Food Delivery System")

# --- 2. DATABASE (Mock Data) ---
menu_db = [
    {"id": 1, "name": "Margherita Pizza", "price": 250.0, "category": "Italian", "status": "active"},
    {"id": 2, "name": "Veg Burger", "price": 99.0, "category": "Fast Food", "status": "active"},
    {"id": 3, "name": "Pasta Carbonara", "price": 320.0, "category": "Italian", "status": "inactive"},
]
orders_db = []

# --- 3. MODELS (Day 2: Pydantic) ---
class MenuCreate(BaseModel):
    name: str = Field(..., min_length=3, max_length=50)
    price: float = Field(..., gt=0)
    category: str
    status: str = "active"

class OrderRequest(BaseModel):
    item_id: int
    quantity: int = Field(..., gt=0)
    customer_name: str

# --- 4. HELPERS (Day 3: Logic) ---
def get_item_by_id(item_id: int):
    return next((item for item in menu_db if item["id"] == item_id), None)

def calculate_total(price: float, qty: int):
    return price * qty

# --- 5. ENDPOINTS (Day 1-6) ---

# [FIXED ROUTES FIRST]

@app.get("/")
def home_route():
    return {"message": "Welcome to the Food Delivery API", "total_items": len(menu_db)}

# Day 6: Combined Browse (Task 20)
@app.get("/menu/browse")
def browse_menu(
    keyword: Optional[str] = None,
    sort_by: str = "price",
    order: str = "asc",
    page: int = 1,
    limit: int = 2
):
    filtered = menu_db
    if keyword:
        filtered = [i for i in menu_db if keyword.lower() in i["name"].lower()]

    rev = True if order == "desc" else False
    sorted_list = sorted(filtered, key=lambda x: x.get(sort_by, 0), reverse=rev)

    start = (page - 1) * limit
    end = start + limit
    return {
        "page": page,
        "total_pages": (len(sorted_list) + limit - 1) // limit,
        "items": sorted_list[start:end]
    }

@app.get("/menu/search")
def search_items(keyword: str):
    results = [i for i in menu_db if keyword.lower() in i["name"].lower()]
    if not results:
        return {"message": f"No items found for {keyword}"}
    return results

@app.post("/menu", status_code=status.HTTP_201_CREATED)
def add_item(item: MenuCreate):
    new_id = len(menu_db) + 1
    new_data = {"id": new_id, **item.dict()}
    menu_db.append(new_data)
    return new_data

@app.post("/orders/place")
def place_order(order: OrderRequest):
    item = get_item_by_id(order.item_id)
    if not item:
        raise HTTPException(status_code=404, detail="Item not found")

    total = calculate_total(item["price"], order.quantity)
    new_order = {
        "order_id": len(orders_db) + 1,
        "customer": order.customer_name,
        "total": total,
        "status": "confirmed"
    }
    orders_db.append(new_order)
    return new_order

# [VARIABLE ROUTES LAST]

@app.get("/menu/{item_id}")
def get_one_item(item_id: int = Path(..., gt=0)):
    item = get_item_by_id(item_id)
    if not item:
        raise HTTPException(status_code=404, detail="Item not found")
    return item

@app.delete("/menu/{item_id}")
def delete_item(item_id: int):
    item = get_item_by_id(item_id)
    if not item:
        raise HTTPException(status_code=404, detail="Item not found")
    if item["status"] == "active":
        raise HTTPException(status_code=400, detail="Cannot delete an active item")
    menu_db.remove(item)
    return {"message": "Deleted successfully"}

In [3]:
from pyngrok import ngrok

# 1. Set Token
ngrok.set_auth_token("3AoIs1ZAu26mNzZPtw9mPxK9LEr_5fMVEsWP9H2wnWATubySh")
ngrok.kill() # Kill old sessions

# 2. Create URL
public_url = ngrok.connect(8000)
print(f"✅ API LIVE: {public_url}")
print(f"🚀 SWAGGER UI: {public_url}/docs")

# 3. Run Server
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [4510]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ API LIVE: NgrokTunnel: "https://isosmotic-costlier-wenona.ngrok-free.dev" -> "http://localhost:8000"
🚀 SWAGGER UI: NgrokTunnel: "https://isosmotic-costlier-wenona.ngrok-free.dev" -> "http://localhost:8000"/docs
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /menu/browse?sort_by=price&order=asc&page=1&limit=2 HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /menu/browse?sort_by=price&order=asc&page=1&limit=2 HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /menu/search?keyword=1 HTTP/1.1" 200 OK
INFO:   

/tmp/ipykernel_4510/3262403173.py:80: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  new_data = {"id": new_id, **item.dict()}


INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /menu HTTP/1.1" 201 Created
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /menu HTTP/1.1" 201 Created
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /orders/place HTTP/1.1" 404 Not Found
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /menu HTTP/1.1" 201 Created
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /menu HTTP/1.1" 201 Created
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "POST /menu HTTP/1.1" 422 Unprocessable Entity
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "GET /menu/3 HTTP/1.1" 200 OK
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "DELETE /menu/1 HTTP/1.1" 400 Bad Request
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "DELETE /menu/8 HTTP/1.1" 400 Bad Request
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "DELETE /menu/100 HTTP/1.1" 404 Not Found
INFO:     2409:40c2:5010:de69:608f:bf31:f047:448c:0 - "DELETE /menu/20 HTTP/1

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4510]
